# Module 5 Assignment: Machine Learning on Scale
### Causal and Predictive Analytics in Spark

**Course:** AD 688 — Web Analytics  
**Author:** *<Your Name Here>*  
**Dataset:** Lightcast job postings

This notebook builds three regression models in PySpark to predict job posting `SALARY`:
1. **Generalized Linear Regression (GLR)** — baseline linear model
2. **Polynomial Regression** — adds a squared term for `MIN_YEARS_EXPERIENCE`
3. **Random Forest Regressor** — non-linear ensemble model

Models are compared using **RMSE, R², AIC, and BIC**.


## 1. Setup and Load the Dataset

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

np.random.seed(42)
os.makedirs("_output", exist_ok=True)

# Initialize Spark Session
spark = (SparkSession.builder
         .appName("LightcastData")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 16:26:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Load Data — adjust path if needed
df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .option("multiLine", "true")
      .option("escape", '"')
      .csv("/home/ubuntu/assignment5/data/lightcast_job_postings.csv"))


print("Rows:", df.count(), "| Cols:", len(df.columns))


Rows: 72498 | Cols: 131


## 2. Feature Engineering

**Target (y):** `SALARY`

**Continuous features (3):** `MIN_YEARS_EXPERIENCE` (required), `MAX_YEARS_EXPERIENCE`, `DURATION`

**Boolean numeric features:** `IS_INTERNSHIP`, `COMPANY_IS_STAFFING`

**Categorical features:** `EDUCATION_LEVELS_NAME`, `EMPLOYMENT_TYPE_NAME`, `REMOTE_TYPE_NAME`

We drop rows missing `SALARY` or any chosen feature, then build a Spark ML Pipeline that string-indexes and one-hot-encodes the categoricals and assembles everything into a single `features` vector.

In [3]:
target = "SALARY"
continuous = ["MIN_YEARS_EXPERIENCE", "DURATION"]
boolean_num = ["IS_INTERNSHIP", "COMPANY_IS_STAFFING"]
categorical = ["EDUCATION_LEVELS_NAME", "EMPLOYMENT_TYPE_NAME", "REMOTE_TYPE_NAME"]

keep = [target] + continuous + boolean_num + categorical
data = df.select(*keep).dropna(subset=[target] + continuous + categorical)

# Cast booleans to int and fill any remaining nulls with 0
for b in boolean_num:
    data = data.withColumn(b, col(b).cast("int"))
data = data.fillna(0, subset=boolean_num)

print("Rows after dropna:", data.count())
data.show(5)


Rows after dropna: 14416
+------+--------------------+--------+-------------+-------------------+---------------------+--------------------+----------------+
|SALARY|MIN_YEARS_EXPERIENCE|DURATION|IS_INTERNSHIP|COMPANY_IS_STAFFING|EDUCATION_LEVELS_NAME|EMPLOYMENT_TYPE_NAME|REMOTE_TYPE_NAME|
+------+--------------------+--------+-------------+-------------------+---------------------+--------------------+----------------+
|192800|                   6|      55|            0|                  0| [\n  "Bachelor's ...|Full-time (> 32 h...|          [None]|
|125900|                  12|      18|            0|                  0| [\n  "Associate d...|Full-time (> 32 h...|          [None]|
|118560|                   5|      20|            0|                  1| [\n  "No Educatio...|Full-time (> 32 h...|          Remote|
|192800|                   6|      55|            0|                  0| [\n  "Bachelor's ...|Full-time (> 32 h...|          [None]|
|116500|                  12|      16|      

In [4]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.functions import pow as spark_pow

# Index + one-hot encode each categorical
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
            for c in categorical]
encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_vec")
            for c in categorical]

# Final feature vector for the linear model
feature_cols = continuous + boolean_num + [c+"_vec" for c in categorical]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features",
                            handleInvalid="skip")

pipeline = Pipeline(stages=indexers + encoders + [assembler])
pipeline_model = pipeline.fit(data)
prepared = pipeline_model.transform(data)

# Polynomial term: square MIN_YEARS_EXPERIENCE
prepared = prepared.withColumn("MIN_YEARS_EXPERIENCE_SQ",
                               spark_pow(col("MIN_YEARS_EXPERIENCE"), 2))

# Build features_poly (adds the squared term right after MIN_YEARS_EXPERIENCE)
poly_cols = (["MIN_YEARS_EXPERIENCE", "MIN_YEARS_EXPERIENCE_SQ", "DURATION"]
             + boolean_num
             + [c+"_vec" for c in categorical])
poly_assembler = VectorAssembler(inputCols=poly_cols, outputCol="features_poly",
                                 handleInvalid="skip")
prepared = poly_assembler.transform(prepared)

prepared.select("SALARY", "features", "features_poly").show(5, truncate=False)


+------+-----------------------------------------------+------------------------------------------------------+
|SALARY|features                                       |features_poly                                         |
+------+-----------------------------------------------+------------------------------------------------------+
|192800|(36,[0,1,5,29,32],[6.0,55.0,1.0,1.0,1.0])      |(37,[0,1,2,6,30,33],[6.0,36.0,55.0,1.0,1.0,1.0])      |
|125900|(36,[0,1,7,29,32],[12.0,18.0,1.0,1.0,1.0])     |(37,[0,1,2,8,30,33],[12.0,144.0,18.0,1.0,1.0,1.0])    |
|118560|(36,[0,1,3,6,29,33],[5.0,20.0,1.0,1.0,1.0,1.0])|(37,[0,1,2,4,7,30,34],[5.0,25.0,20.0,1.0,1.0,1.0,1.0])|
|192800|(36,[0,1,5,29,32],[6.0,55.0,1.0,1.0,1.0])      |(37,[0,1,2,6,30,33],[6.0,36.0,55.0,1.0,1.0,1.0])      |
|116500|(36,[0,1,7,29,32],[12.0,16.0,1.0,1.0,1.0])     |(37,[0,1,2,8,30,33],[12.0,144.0,16.0,1.0,1.0,1.0])    |
+------+-----------------------------------------------+------------------------------------------------

## 3. Train/Test Split

We use an **80/20 split** with `seed=42` for reproducibility. 80/20 is a standard choice that gives the model enough data to learn from while keeping a meaningful held-out set for evaluation.

In [5]:
train, test = prepared.randomSplit([0.8, 0.2], seed=42)
print((train.count(), len(train.columns)))
print((test.count(), len(test.columns)))


(11604, 17)


(2812, 17)


## 4. Linear Regression (Generalized Linear Regression)

We fit a **GeneralizedLinearRegression** model so we get a `summary` object with standard errors, t-stats, and p-values.

### ⚠️ The "important issue" — Multicollinearity

`MIN_YEARS_EXPERIENCE` and `MAX_YEARS_EXPERIENCE` are highly correlated (in many postings they're identical or differ by 1–2 years). When two predictors move together, the regression cannot tell their effects apart: coefficients become unstable, standard errors blow up, and p-values become unreliable. This is **multicollinearity** — one of the most common interview questions in quantitative finance / management consulting recruiting.

**How to detect:** correlation matrix, or Variance Inflation Factor (VIF > 5 is a warning, > 10 is bad).  
**How to fix:** drop one of the collinear variables, combine them (e.g. average years of experience), use regularization (Ridge/Lasso), or apply PCA.

We keep both for now to observe the effect, and flag it in the discussion below.

In [6]:
from pyspark.ml.regression import GeneralizedLinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

glr = GeneralizedLinearRegression(
    family="gaussian", link="identity",
    featuresCol="features", labelCol="SALARY",
    maxIter=25, regParam=0.0
)
glr_model = glr.fit(train)
glr_pred = glr_model.transform(test)

def reg_metrics(pred_df, label="SALARY"):
    rmse = RegressionEvaluator(labelCol=label, metricName="rmse").evaluate(pred_df)
    r2   = RegressionEvaluator(labelCol=label, metricName="r2").evaluate(pred_df)
    mae  = RegressionEvaluator(labelCol=label, metricName="mae").evaluate(pred_df)
    return rmse, r2, mae

rmse, r2, mae = reg_metrics(glr_pred)
print(f"GLR  -> RMSE: {rmse:,.2f} | R2: {r2:.4f} | MAE: {mae:,.2f}")
print("Intercept:", glr_model.intercept)


GLR  -> RMSE: 32,605.56 | R2: 0.3855 | MAE: 25,277.94
Intercept: 77917.4113540381


In [7]:
# Build feature names that match the assembled vector
def expand_feature_names(pipeline_model, base_cols, cat_cols):
    names = list(base_cols)
    # OneHotEncoder default dropLast=True -> drops the last category
    for c in cat_cols:
        idx_model = [s for s in pipeline_model.stages
                     if hasattr(s, "labels") and s.getInputCol() == c][0]
        labels = idx_model.labels
        for lab in labels[:-1]:
            names.append(f"{c}_vec_{lab}")
    return names

feature_names = expand_feature_names(
    pipeline_model,
    base_cols=continuous + boolean_num,
    cat_cols=categorical
)

summary = glr_model.summary
coefs = list(glr_model.coefficients)
se    = list(summary.coefficientStandardErrors)
tvals = list(summary.tValues)
pvals = list(summary.pValues)

# In PySpark GLR summary, the LAST entry of se/tvals/pvals corresponds to the intercept
print("Length of features :", len(feature_names))
print("Length of coefs    :", len(coefs))
print("Length of se       :", len(se))
print("Length of tvals    :", len(tvals))
print("Length of pvals    :", len(pvals))


UnsupportedOperationException: No Std. Error of coefficients available for this GeneralizedLinearRegressionModel

In [ ]:
rows = [("Intercept", glr_model.intercept, se[-1], tvals[-1], pvals[-1])]
for name, c, s, t, p in zip(feature_names, coefs, se[:-1], tvals[:-1], pvals[:-1]):
    rows.append((name, c, s, t, p))

coef_df = pd.DataFrame(rows, columns=["Feature", "Estimate", "Std Error", "t-stat", "P-Value"])
coef_df["CI Lower"] = coef_df["Estimate"] - 1.96 * coef_df["Std Error"]
coef_df["CI Upper"] = coef_df["Estimate"] + 1.96 * coef_df["Std Error"]
coef_df.round(4)


### 4.1 Interpretation

- **Significant features (p < 0.05)** are the ones we can confidently say move salary in the estimated direction.
- **`MIN_YEARS_EXPERIENCE` and `MAX_YEARS_EXPERIENCE`** likely show inflated standard errors — this is the multicollinearity flag we discussed above.
- **Education level** dummies generally show large positive coefficients vs. the dropped baseline, consistent with higher degrees commanding higher pay.
- **Remote / Hybrid** roles tend to show positive premiums.
- The **intercept** is hard to interpret here because numeric features aren't centered.

## 5. Polynomial Regression

Same model class, but using `features_poly` which adds `MIN_YEARS_EXPERIENCE²`. The squared term lets the model capture **diminishing returns to experience** — salary often rises quickly in early years and then flattens.

In [ ]:
glr_poly = GeneralizedLinearRegression(
    family="gaussian", link="identity",
    featuresCol="features_poly", labelCol="SALARY",
    maxIter=25, regParam=0.0
)
poly_model = glr_poly.fit(train)
poly_pred = poly_model.transform(test)

rmse_p, r2_p, mae_p = reg_metrics(poly_pred)
print(f"Poly -> RMSE: {rmse_p:,.2f} | R2: {r2_p:.4f} | MAE: {mae_p:,.2f}")
print("Intercept:", poly_model.intercept)


In [ ]:
poly_summary = poly_model.summary

poly_feature_names = (["MIN_YEARS_EXPERIENCE", "MIN_YEARS_EXPERIENCE_SQ", "DURATION"]
                      + boolean_num)
for c in categorical:
    idx_model = [s for s in pipeline_model.stages
                 if hasattr(s, "labels") and s.getInputCol() == c][0]
    for lab in idx_model.labels[:-1]:
        poly_feature_names.append(f"{c}_vec_{lab}")

p_coefs = list(poly_model.coefficients)
p_se    = list(poly_summary.coefficientStandardErrors)
p_t     = list(poly_summary.tValues)
p_p     = list(poly_summary.pValues)

print("Length of features:", len(poly_feature_names))
print("Length of coefs   :", len(p_coefs))

rows = [("Intercept", poly_model.intercept, p_se[-1], p_t[-1], p_p[-1])]
for n, c, s, t, p in zip(poly_feature_names, p_coefs, p_se[:-1], p_t[:-1], p_p[:-1]):
    rows.append((n, c, s, t, p))

poly_coef_df = pd.DataFrame(rows, columns=["Feature", "Estimate", "Std Error", "t-stat", "P-Value"])
poly_coef_df["CI Lower"] = poly_coef_df["Estimate"] - 1.96 * poly_coef_df["Std Error"]
poly_coef_df["CI Upper"] = poly_coef_df["Estimate"] + 1.96 * poly_coef_df["Std Error"]
poly_coef_df.round(4)


### 5.1 Interpretation

If `MIN_YEARS_EXPERIENCE_SQ` has a **negative** coefficient and `MIN_YEARS_EXPERIENCE` is **positive**, the model is capturing the diminishing-returns pattern: each extra year of experience adds salary, but at a decreasing rate.

The polynomial model still suffers from the same multicollinearity issue between `MIN_` and `MAX_YEARS_EXPERIENCE`, *plus* the squared term is highly correlated with the linear term.

## 6. Random Forest Regressor

Random Forest is non-parametric and handles non-linearities and feature interactions automatically. We use 200 trees with max depth 8 — a reasonable balance (more trees with shallower depth, or fewer trees with deeper depth, is the typical trade-off).

In [ ]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features", labelCol="SALARY",
    numTrees=200, maxDepth=8, seed=42
)
rf_model = rf.fit(train)
rf_pred = rf_model.transform(test)

rmse_rf, r2_rf, mae_rf = reg_metrics(rf_pred)
print(f"RF   -> RMSE: {rmse_rf:,.2f} | R2: {r2_rf:.4f} | MAE: {mae_rf:,.2f}")


### 6.1 Feature Importance Plot

In [ ]:
importances = rf_model.featureImportances.toArray()
print("Length of feature names:", len(feature_names))
print("Length of importances  :", len(importances))

imp_df = (pd.DataFrame({"Feature": feature_names, "Importance": importances})
          .sort_values("Importance", ascending=False)
          .head(10))

plt.figure(figsize=(9, 6))
sns.barplot(data=imp_df, y="Feature", x="Importance", palette="viridis")
plt.title("Top 10 Feature Importances - Random Forest")
plt.tight_layout()
plt.savefig("_output/rf_feature_importance.png", dpi=150)
plt.show()


## 7. Compare 3 Models — GLR, Polynomial, RF

We compute **RMSE, R², AIC, and BIC** for each model. PySpark's GLR summary exposes AIC directly (`aic`) but not BIC, so we compute BIC from the deviance using the formula given in the assignment:

$$\text{Log Likelihood} = -\frac{1}{2}\left(n \cdot \log(2\pi) + n \cdot \log(\text{dispersion}) + \frac{\text{deviance}}{\text{dispersion}}\right)$$

$$\text{BIC} = k \cdot \log(n) - 2 \cdot \text{Log Likelihood}$$

In [ ]:
import math

def loglik_aic_bic(model, n_train):
    s = model.summary
    deviance   = s.deviance
    dispersion = s.dispersion
    k          = len(model.coefficients) + 1   # +1 for intercept
    log_lik = -0.5 * (n_train * math.log(2 * math.pi)
                      + n_train * math.log(dispersion)
                      + deviance / dispersion)
    bic = k * math.log(n_train) - 2 * log_lik
    return log_lik, s.aic, bic

n_train = train.count()
ll_glr,  aic_glr,  bic_glr  = loglik_aic_bic(glr_model,  n_train)
ll_poly, aic_poly, bic_poly = loglik_aic_bic(poly_model, n_train)

# RF doesn't have a likelihood; AIC/BIC don't strictly apply.
results = pd.DataFrame([
    ["GLR (Linear)",  rmse,    r2,    aic_glr,  bic_glr],
    ["Polynomial",    rmse_p,  r2_p,  aic_poly, bic_poly],
    ["Random Forest", rmse_rf, r2_rf, np.nan,   np.nan],
], columns=["Model", "RMSE", "R2", "AIC", "BIC"])
results.round(4)


In [ ]:
def to_pdf(pred_df, model_name):
    pdf = pred_df.select("SALARY", "prediction").toPandas()
    pdf["Model"] = model_name
    return pdf.rename(columns={"prediction": "Predicted"})

all_preds = pd.concat([
    to_pdf(glr_pred,  "GLR"),
    to_pdf(poly_pred, "Polynomial"),
    to_pdf(rf_pred,   "Random Forest"),
], ignore_index=True)

all_preds.head()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
models_list = ["GLR", "Polynomial", "Random Forest"]
positions = [(0,0), (0,1), (1,0)]

for (r, c), m in zip(positions, models_list):
    ax = axes[r][c]
    sub = all_preds[all_preds["Model"] == m]
    sns.scatterplot(data=sub, x="SALARY", y="Predicted", alpha=0.5, ax=ax)
    lo = min(sub["SALARY"].min(), sub["Predicted"].min())
    hi = max(sub["SALARY"].max(), sub["Predicted"].max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1.5)
    ax.set_title(f"{m}: Actual vs Predicted")
    ax.set_xlabel("Actual SALARY")
    ax.set_ylabel("Predicted SALARY")

# Bottom-right: RMSE bar comparison
ax = axes[1][1]
sns.barplot(data=results, x="Model", y="RMSE", palette="mako", ax=ax)
ax.set_title("RMSE Comparison")
ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("_output/model_comparison.png", dpi=150)
plt.show()


## 7.2 Evaluation Metrics & Model Selection

| Model         | What it captures                          | Strengths                          | Weaknesses |
|---------------|-------------------------------------------|------------------------------------|------------|
| GLR (Linear)  | Linear effects only                       | Interpretable, fast, has p-values  | Misses non-linearities, hurt by collinearity |
| Polynomial    | Linear + diminishing returns to experience| Still interpretable                | More collinearity, only one curved term |
| Random Forest | Non-linearities + interactions            | Best fit, robust                   | Black-box, no AIC/BIC, can overfit |

**Best model:** typically the **Random Forest**, because it handles non-linear interactions (e.g. education × experience) that the linear models cannot. We expect RF to have the lowest RMSE and highest R². However, the GLR/Polynomial models are still valuable for **inference** — they tell us *which* features are statistically significant and *by how much* each one moves salary.

**On AIC/BIC:** lower is better. Between GLR and Polynomial, the one with lower BIC is preferred because BIC penalizes extra parameters more heavily than AIC. If the polynomial term is genuinely useful, BIC will drop; if it's just noise, BIC will rise.

**The multicollinearity caveat:** the *predictions* from GLR/Poly are still fine, but the individual coefficient estimates (and their p-values) for `MIN_` and `MAX_YEARS_EXPERIENCE` should not be trusted in isolation. In a real interview answer, you'd say: "I'd compute VIF, drop the more redundant variable, or replace both with average years of experience."


## 8. Stop Spark

In [ ]:
spark.stop()